# Business Financial Crisis Prediction AI System
## Early Warning System for Corporate Financial Distress

This end-to-end notebook implements a prototype Early Warning System (EWS) for financial distress prediction across companies.

### Key Notebook Sections:
1. **Synthetic Financial Panel Dataset Generation** (800 companies across 5 fiscal years: 2020–2024, 4,000 records)
2. **Feature Preprocessing Pipeline** (Zero data-leakage scaling & encoding)
3. **Model Benchmark & Class Imbalance Handling** (Logistic Regression, Random Forest, XGBoost + SMOTE)
4. **Temporal Out-Of-Time (OOT) Cross-Validation** (Train: 2020–2022, Test: 2023–2024)
5. **Explainability & SHAP Analysis** (Global importances & plain business English local attributions)
6. **Robustness & Stress Testing Engine** (Simulating accounting noise & missing financial statements)
7. **Cost-Sensitive Decision Threshold Tuning** (Optimizing for high crisis recall)

In [ ]:
# Step 0: Imports and Environment Setup
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generator import generate_financial_dataset
from src.features import FinancialFeaturePipeline, NUMERICAL_FEATURES
from src.models import train_all_models, predict_crisis_probabilities, find_optimal_threshold, assign_risk_tier
from src.explainability import get_global_feature_importance, explain_company_risk
from src.evaluation import run_out_of_time_evaluation, run_stress_test, evaluate_predictions

print("Libraries and project modules loaded successfully!")

## 1. Dataset Generation & Exploratory Data Analysis (EDA)
We generate a longitudinal panel dataset of 800 companies tracked over 5 years (2020 to 2024).

In [ ]:
df = generate_financial_dataset(n_companies=800, years=(2020, 2021, 2022, 2023, 2024), seed=42)
print(f"Dataset Shape: {df.shape}")
display(df.head(5))

print("\nTarget Class Breakdown (Distress Tier):")
print(df["distress_tier"].value_counts(normalize=True).apply(lambda x: f"{x*100:.2f}%"))

### Financial Ratio Distributions across Risk Tiers

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
sns.boxplot(data=df, x="distress_tier", y="current_ratio", ax=axes[0,0], palette="Set2")
axes[0,0].set_title("Current Ratio (Liquidity) by Distress Tier")

sns.boxplot(data=df, x="distress_tier", y="debt_to_equity", ax=axes[0,1], palette="Set2")
axes[0,1].set_title("Debt-to-Equity (Leverage) by Distress Tier")

sns.boxplot(data=df, x="distress_tier", y="interest_coverage_ratio", ax=axes[1,0], palette="Set2")
axes[1,0].set_title("Interest Coverage Ratio (Solvency) by Distress Tier")

sns.boxplot(data=df, x="distress_tier", y="roa", ax=axes[1,1], palette="Set2")
axes[1,1].set_title("ROA (Profitability) by Distress Tier")
plt.tight_layout()
plt.show()

## 2. Feature Engineering & Preprocessing Pipeline
We fit our transformers exclusively on historical training data to prevent data leakage.

In [ ]:
pipeline = FinancialFeaturePipeline()
X_transformed = pipeline.fit_transform(df)
y = df["is_crisis"].values
print(f"Transformed Feature Matrix Shape: {X_transformed.shape}")
print(f"Features included: {pipeline.get_feature_names()[:10]}...")

## 3. Temporal Out-Of-Time (OOT) Model Benchmark
We split data by Fiscal Years:
- **Training Set**: 2020 – 2022 (2,400 company-years)
- **Out-of-Time Test Set**: 2023 – 2024 (1,600 company-years)

We benchmark **Logistic Regression**, **Random Forest**, and **XGBoost** with SMOTE resampling.

In [ ]:
oot_results = {}
for m_name in ["Logistic Regression", "Random Forest", "XGBoost"]:
    res = run_out_of_time_evaluation(df, train_years=(2020, 2021, 2022), test_years=(2023, 2024), model_name=m_name, use_smote=True)
    oot_results[m_name] = res

benchmark_df = pd.DataFrame([
    {
        "Model": m_name,
        "OOT ROC-AUC": res["oot_test_metrics"]["roc_auc"],
        "Crisis Recall": f"{res['oot_test_metrics']['recall']*100:.1f}%",
        "Precision": f"{res['oot_test_metrics']['precision']*100:.1f}%",
        "F1-Score": res["oot_test_metrics"]["f1"],
        "Decision Threshold": res["optimal_threshold"]
    }
    for m_name, res in oot_results.items()
])
display(benchmark_df)

## 4. Model Explainability & Feature Attributions
We extract global feature importances and local plain business English explanations for an individual company.

In [ ]:
best_pipeline = oot_results["XGBoost"]["fitted_pipeline"]
best_model = oot_results["XGBoost"]["fitted_model"]
feat_names = best_pipeline.get_feature_names()

df_imp = get_global_feature_importance(best_model, feat_names, top_n=10)
plt.figure(figsize=(10, 5))
plt.barh(df_imp["human_name"], df_imp["importance_pct"], color="teal")
plt.gca().invert_yaxis()
plt.title("Top 10 Global Feature Importances (XGBoost)")
plt.xlabel("Importance (%)")
plt.show()

print("\n--- Local Plain-Language Explanation for Sample Company ---")
sample_record = df.iloc[0].to_dict()
X_sample = best_pipeline.transform(df.iloc[0:1])
exps = explain_company_risk(best_model, X_sample, feat_names, sample_record)
for exp in exps:
    print(f"• {exp['narrative']} (SHAP impact: {exp['shap_value']:+.3f})")

## 5. Noise & Missing Data Robustness Stress Testing
We simulate audit restatements (Gaussian noise) and missing financial statements.

In [ ]:
test_df = df[df["year"].isin([2023, 2024])].copy()
stress_results = run_stress_test(test_df, best_pipeline, best_model, optimal_threshold=oot_results["XGBoost"]["optimal_threshold"])
display(stress_results.head(10))

## 6. Threshold Optimization & Cost Matrix Tradeoff Analysis
False Negatives (missing a crisis) are assumed to be 10x more expensive than False Positives.

In [ ]:
X_test = best_pipeline.transform(test_df)
y_test = test_df["is_crisis"].values
probs_test = predict_crisis_probabilities(best_model, X_test)

thresholds = np.linspace(0.1, 0.9, 81)
costs, recalls, precisions = [], [], []
for th in thresholds:
    m = evaluate_predictions(y_test, probs_test, threshold=th)
    c = (m["fn"] * 10.0) + (m["fp"] * 1.0)
    costs.append(c)
    recalls.append(m["recall"])
    precisions.append(m["precision"])

plt.figure(figsize=(10, 5))
plt.plot(thresholds, recalls, label="Crisis Recall", color="green", lw=2)
plt.plot(thresholds, precisions, label="Precision", color="blue", lw=2)
plt.axvline(x=oot_results["XGBoost"]["optimal_threshold"], color="red", linestyle="--", label=f"Optimal Threshold ({oot_results['XGBoost']['optimal_threshold']:.2f})")
plt.title("Precision / Recall Tradeoff & Optimal Decision Threshold")
plt.xlabel("Probability Threshold")
plt.ylabel("Score")
plt.legend()
plt.grid(True)
plt.show()